In [ ]:
!pip install -q transformers accelerate gradio

In [ ]:
import torch
import gradio as gr

from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype="auto"
)

model.to(device)
model.eval()

print("Model loaded successfully!")

Using device: cpu


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
def generate_content(
    topic,
    content_type,
    tone,
    length,
    temperature=0.7
):

    length_map = {
        "Short": 150,
        "Medium": 300,
        "Long": 500
    }

    max_tokens = length_map[length]

    prompt = f"""Write a high-quality {content_type} about {topic}.

Tone: {tone}
Length: approximately {max_tokens} words.

Make the content clear, engaging, well-structured, and relevant.
Return only the final content.
"""

    # Tokenize normally
    encoded = tokenizer(
        prompt,
        return_tensors="pt"
    )

    input_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)

    # Generate text
    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_tokens,
            temperature=float(temperature),
            do_sample=True,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    # Remove the original prompt
    generated_tokens = outputs[0, input_ids.shape[-1]:]

    # Decode
    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return generated_text.strip()

In [ ]:
result = generate_content(
    topic="The impact of Artificial Intelligence on education",
    content_type="Blog Post",
    tone="Professional",
    length="Medium",
    temperature=0.7
)

print(result)

Do not include any existing information in your response. Provide only the new content.
Include a minimum of two academic sources that back up your points. Use appropriate formatting (e.g., APA or MLA) for citations and references.

The impact of artificial intelligence on education

Artificial intelligence is rapidly changing the way we learn, teach, and interact with technology. As AI becomes more advanced and accessible, it has the potential to transform the field of education by enabling smarter, more efficient learning experiences. This blog post will explore the ways in which AI can improve education, including its impact on teaching methods, student engagement, and assessment processes.

One of the most significant benefits of AI in education is its ability to personalize learning. Traditional education often relies on rote memorization and one-size-fits-all approaches, which may not be effective for all students. AI-powered educational technologies, such as adaptive learning pl

In [ ]:
import gradio as gr

def generate_from_ui(
    topic,
    content_type,
    tone,
    length,
    temperature
):
    return generate_content(
        topic=topic,
        content_type=content_type,
        tone=tone,
        length=length,
        temperature=temperature
    )


with gr.Blocks(title="AI Content Writer") as demo:

    gr.Markdown(
        """
        # ✍️ AI Content Writer
        ### Generate high-quality content using a pretrained Transformer model
        """
    )

    with gr.Row():

        with gr.Column():

            topic = gr.Textbox(
                label="Topic / Prompt",
                placeholder="Enter the topic you want to write about...",
                lines=5
            )

            content_type = gr.Dropdown(
                choices=[
                    "Blog Post",
                    "Article",
                    "Essay",
                    "Social Media Post",
                    "Product Description",
                    "Marketing Copy"
                ],
                value="Blog Post",
                label="Content Type"
            )

            tone = gr.Dropdown(
                choices=[
                    "Professional",
                    "Friendly",
                    "Informative",
                    "Persuasive",
                    "Creative",
                    "Casual"
                ],
                value="Professional",
                label="Tone"
            )

            length = gr.Dropdown(
                choices=[
                    "Short",
                    "Medium",
                    "Long"
                ],
                value="Medium",
                label="Length"
            )

            temperature = gr.Slider(
                minimum=0.1,
                maximum=1.2,
                value=0.7,
                step=0.1,
                label="Temperature / Creativity"
            )

            generate_btn = gr.Button(
                "✨ Generate Content",
                variant="primary"
            )

            clear_btn = gr.ClearButton(
                value="Clear"
            )

        with gr.Column():

           output = gr.Textbox(
    label="Generated Content",
    lines=20
)


    generate_btn.click(
        fn=generate_from_ui,
        inputs=[
            topic,
            content_type,
            tone,
            length,
            temperature
        ],
        outputs=output
    )

    clear_btn.add(
        [topic, output]
    )


demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5195122c73eeda0963.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
